In [0]:
display(spark.sql("SHOW TABLES IN `4_prod`.staging"))

In [0]:
# ==============================================================================
# CONTROL VARIABLES
# ==============================================================================

# Which environment to operate on: "dev" or "prod"
# "dev": reads from 8_dev.staging, writes to 8_dev.raw, uses 8_dev.tmp for lookups
# "prod": reads from 4_prod.staging, writes to 4_prod.raw, uses 4_prod.tmp for lookups
TARGET_ENV = "prod"

In [0]:
# SECTION 1: Configuration
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
from delta.tables import DeltaTable
import time

if TARGET_ENV == "dev":
    CATALOG = "8_dev"
else:
    CATALOG = "4_prod"

RAW_SCHEMA = "raw"
STAGING_SCHEMA = "staging"
TMP_SCHEMA = "tmp"
ABANDONED_SCHEMA = "staging_abandoned"
LOG_CATALOG = "6_mgmt"
LOG_SCHEMA = "logs"

spark.sql(f"USE CATALOG {CATALOG}")

RUN_TS = datetime.now()

LOOKUP_RETENTION_DAYS = 21
METADATA_FRESHNESS_DAYS = 30
INITIAL_LOOKBACK_VERSIONS = 25
FILTER_BHRUT = True
BACKWINDOW_DAYS = 7
FALLBACK_LOOKBACK_HOURS = 48
# After this many unsuccessful classification attempts, a staging row is moved
# to {ABANDONED_SCHEMA}.{tbl} and deleted from staging so it stops consuming
# compute on every run.
MAX_CLASSIFICATION_ATTEMPTS = 30

# All table FQNs built from CATALOG
TRUST_MAP_TBL = f"{CATALOG}.{TMP_SCHEMA}.org_to_trust_map"
ORG_HIST_TBL = f"{CATALOG}.{TMP_SCHEMA}.org_mapping_history"
HUB_TBL = f"{CATALOG}.{TMP_SCHEMA}.sw_mapping_hub"
ENC_ORG_TBL = f"{CATALOG}.{TMP_SCHEMA}.sw_enc_org_map"
CHANGED_ENC_TBL = f"{CATALOG}.{TMP_SCHEMA}.sw_changed_encounters"
CONTROL_TBL = f"{CATALOG}.{TMP_SCHEMA}.incr_updt_trust_control"
FLAG_TBL = f"{LOG_CATALOG}.{LOG_SCHEMA}.organization_flags"
AUDIT_SUMMARY_TBL = f"{LOG_CATALOG}.{LOG_SCHEMA}.trust_audit_summary"
CLASSIFICATION_LOG_TBL = f"{LOG_CATALOG}.{LOG_SCHEMA}.trust_classification_log"

BARTS_ORGS = [
    873843, 8367658, 669849, 9073614, 2681833, 4401825, 3203824, 2681830,
    8061679, 669848, 8467812, 2681824, 2619824, 2681827, 3203825, 691988,
    3125827, 8061682, 8061694, 2641824, 2641827, 669847, 8056759, 8061685,
    2641830, 3201824, 691989, 669845, 669843, 8061691, 669846, 3199824,
    669850, 6333825, 669844, 8397458, 8152502, 671843, 613843
]
BHRUT_ORGS = [9161976, 9163579, 9161983, 723896, 9161987, 9161988, 9163583, 9161989]

KEY_LOOKUPS = [
    ("SURG_CASE_ID", f"{CATALOG}.{RAW_SCHEMA}.mill_surgical_case", "ENCNTR_ID"),
    ("SURG_CASE_PROC_ID", f"{CATALOG}.{RAW_SCHEMA}.mill_surg_case_procedure", "NULL"),
    ("DCP_FORMS_ACTIVITY_ID", f"{CATALOG}.{RAW_SCHEMA}.mill_dcp_forms_activity", "ENCNTR_ID"),
    ("EPISODE_ID", f"{CATALOG}.{RAW_SCHEMA}.mill_episode_encntr_reltn", "ENCNTR_ID"),
    ("ORDER_ID", f"{CATALOG}.{RAW_SCHEMA}.mill_orders", "ENCNTR_ID"),
    ("PROBLEM_ID", f"{CATALOG}.{RAW_SCHEMA}.mill_problem", "COALESCE(ORIGINATING_ENCNTR_ID, UPDATE_ENCNTR_ID)"),
    ("SCH_EVENT_ID", f"{CATALOG}.{RAW_SCHEMA}.mill_sch_event_patient", "ENCNTR_ID"),
    ("SCHEDULE_ID", f"{CATALOG}.{RAW_SCHEMA}.mill_sch_schedule", "ENCNTR_ID"),
    ("IM_STUDY_ID", f"{CATALOG}.{RAW_SCHEMA}.mill_im_study", "ENCNTR_ID"),
    ("IM_ACQUIRED_STUDY_ID", f"{CATALOG}.{RAW_SCHEMA}.mill_im_acquired_study", "NULL"),
    ("CV_PROC_ID", f"{CATALOG}.{RAW_SCHEMA}.mill_cv_proc", "ENCNTR_ID")
]

CHAINED_DEPENDENCIES = {
    "IM_STUDY_ID": ("mill_im_acquired_study", "MATCHED_STUDY_ID", "IM_ACQUIRED_STUDY_ID"),
    "SURG_CASE_ID": ("mill_surg_case_procedure", "SURG_CASE_ID", "SURG_CASE_PROC_ID")
}

TABLES_SKIP_NULL_TRUST_LOGGING = {
    "MILL_PERSON_ORG_RELTN", "MILL_ORG_ORG_RELTN", "MILL_ORGANIZATION_ALIAS",
    "MILL_ORGANIZATION", "MILL_PRSNL_ORG_RELTN", "MILL_ORG_TYPE_RELTN",
    "MILL_SCH_LOCATION", "MILL_SCH_EVENT", "MILL_SCH_APPT",
    "MILL_SCH_EVENT_ALIAS", "MILL_SCH_SCHEDULE"
}

SOURCE_OF_TRUTH_TABLES = {
    "MILL_CLINICAL_EVENT", "MILL_ENCOUNTER", "MILL_ORDERS",
    "MILL_SURGICAL_CASE", "MILL_DCP_FORMS_ACTIVITY",
    "MILL_EPISODE_ENCNTR_RELTN", "MILL_SCH_EVENT_PATIENT",
    "MILL_SCH_SCHEDULE", "MILL_IM_STUDY", "MILL_CV_PROC",
    "MILL_PROBLEM"
}

print(f"Trust Classification Incremental pipeline initialized at {RUN_TS}")
print(f"TARGET_ENV: {TARGET_ENV}")
print(f"Catalog: {CATALOG} | Raw: {RAW_SCHEMA} | Staging: {STAGING_SCHEMA} | Tmp: {TMP_SCHEMA}")
print(f"Classification log always writes to: {CLASSIFICATION_LOG_TBL}")

In [0]:
# SECTION 2: Helper Functions

def time_op(name, fn):
    """Execute a function and print its elapsed time."""
    t0 = time.time()
    result = fn()
    elapsed = time.time() - t0
    print(f"  [{name}] completed in {elapsed:.1f}s")
    return result



def log_status(table_name, status, reason=""):
    """Colored status logging for pipeline operations."""
    colors = {
        "QUEUE": "\033[94m",   # blue
        "PROC":  "\033[92m",   # green
        "ERR":   "\033[91m",   # red
        "SKIP":  "\033[93m",   # yellow
        "INFO":  "\033[96m",   # cyan
        "INIT":  "\033[95m",   # magenta
    }
    reset = "\033[0m"
    color = colors.get(status, "")
    ts = datetime.now().strftime("%H:%M:%S")
    msg = f"{color}[{ts}] [{status}] {table_name}{reset}"
    if reason:
        msg += f" - {reason}"
    print(msg)


def get_table_columns(fqn):
    """Return a set of uppercase column names for the given fully-qualified table."""
    try:
        cols = spark.table(fqn).columns
        return {c.upper() for c in cols}
    except Exception:
        return set()


def can_delete_bhrut(table_upper):
    """Check if BHRUT rows can be deleted from this table (i.e. not in the skip list)."""
    return table_upper not in TABLES_SKIP_NULL_TRUST_LOGGING


def pick_time_col(fqn):
    """Pick the best timestamp column for fallback filtering: ADC_UPDT > UPDT_DT_TM > CLINSIG_UPDT_DT_TM."""
    cols = get_table_columns(fqn)
    for candidate in ["ADC_UPDT", "UPDT_DT_TM", "CLINSIG_UPDT_DT_TM"]:
        if candidate in cols:
            return candidate
    return None


def determine_pk_column(table_name, cols, raw_cols=None):
    """
    Determine the primary key column for a given table.
    Uses a mapping of known table->PK, with fallback heuristics.

    When raw_cols is supplied, only candidates present in raw are returned, so
    staging-only columns (ENCNTR_ID/ORGANIZATION_ID bolted on for tagging) can
    never be picked for tables whose raw target doesn't carry them.
    """
    tbl_upper = table_name.upper()
    cols_upper = {c.upper() for c in cols}
    raw_cols_upper = {c.upper() for c in raw_cols} if raw_cols is not None else None

    # Explicit PK mapping for known tables
    pk_map = {
        "MILL_CLINICAL_EVENT": "EVENT_ID",
        "MILL_ENCOUNTER": "ENCNTR_ID",
        "MILL_ORDERS": "ORDER_ID",
        "MILL_PERSON": "PERSON_ID",
        "MILL_PERSON_ALIAS": "PERSON_ALIAS_ID",
        "MILL_ENCNTR_ALIAS": "ENCNTR_ALIAS_ID",
        "MILL_ENCOUNTER_ALIAS": "ENCNTR_ALIAS_ID",
        "MILL_SURGICAL_CASE": "SURG_CASE_ID",
        "MILL_SURG_CASE_PROCEDURE": "SURG_CASE_PROC_ID",
        "MILL_DCP_FORMS_ACTIVITY": "DCP_FORMS_ACTIVITY_ID",
        "MILL_EPISODE_ENCNTR_RELTN": "EPISODE_ID",
        "MILL_PROBLEM": "PROBLEM_ID",
        "MILL_SCH_EVENT_PATIENT": "SCH_EVENT_ID",
        "MILL_SCH_SCHEDULE": "SCHEDULE_ID",
        "MILL_IM_STUDY": "IM_STUDY_ID",
        "MILL_IM_ACQUIRED_STUDY": "IM_ACQUIRED_STUDY_ID",
        "MILL_CV_PROC": "CV_PROC_ID",
        "MILL_PERSON_ORG_RELTN": "PERSON_ORG_RELTN_ID",
        "MILL_ORG_ORG_RELTN": "ORG_ORG_RELTN_ID",
        "MILL_ORGANIZATION_ALIAS": "ORGANIZATION_ALIAS_ID",
        "MILL_ORGANIZATION": "ORGANIZATION_ID",
        "MILL_PRSNL_ORG_RELTN": "PRSNL_ORG_RELTN_ID",
        "MILL_ORG_TYPE_RELTN": "ORG_TYPE_RELTN_ID",
        "MILL_SCH_LOCATION": "SCH_LOCATION_ID",
        "MILL_SCH_EVENT": "SCH_EVENT_ID",
        "MILL_SCH_APPT": "SCH_APPT_ID",
        "MILL_SCH_EVENT_ALIAS": "SCH_EVENT_ALIAS_ID",
        "MILL_DIAGNOSIS": "DIAGNOSIS_ID",
        "MILL_PROCEDURE": "PROCEDURE_ID",
        "MILL_ALLERGY": "ALLERGY_INSTANCE_ID",
        "MILL_BLOB_CONTENT": "EVENT_ID",
        "MILL_CE_BLOB": "EVENT_ID",
        "MILL_CE_BLOB_RESULT": "EVENT_ID",
        "MILL_CE_CODED_RESULT": "EVENT_ID",
        "MILL_CE_DATE_RESULT": "EVENT_ID",
        "MILL_CE_MED_RESULT": "EVENT_ID",
        "MILL_CE_STRING_RESULT": "EVENT_ID",
        "MILL_CLINICAL_EVENT_ACTION": "ACTION_EVENT_ID",
        "MILL_ENCOUNTER_COMBINE": "COMBINE_ID",
        "MILL_ENCNTR_LOC_HIST": "ENCNTR_LOC_HIST_ID",
        "MILL_ENCNTR_PLAN_RELTN": "ENCNTR_PLAN_RELTN_ID",
        "MILL_ENCNTR_PRSNL_RELTN": "ENCNTR_PRSNL_RELTN_ID",
        "MILL_ORDER_ACTION": "ORDER_ID",
        "MILL_ORDER_COMMENT": "ORDER_ID",
        "MILL_ORDER_DETAIL": "ORDER_ID",
        "MILL_ORDER_INGREDIENT": "ORDER_ID",
        "MILL_PERSON_COMBINE": "COMBINE_ID",
        "MILL_PERSON_NAME": "PERSON_NAME_ID",
        "MILL_PERSON_PATIENT": "PERSON_ID",
        "MILL_PERSON_PRSNL_RELTN": "PERSON_PRSNL_R_ID",
        "MILL_PRSNL": "PERSON_ID",
        "MILL_PRSNL_ALIAS": "PRSNL_ALIAS_ID",
        "MILL_ADDRESS": "ADDRESS_ID",
        "MILL_PHONE": "PHONE_ID",
        "MILL_CODE_VALUE": "CODE_VALUE",
        "MILL_CODE_VALUE_ALIAS": "ALIAS_ID",
        "MILL_CODE_VALUE_SET": "CODE_SET",
        "MILL_IM_ACQUIRED_STUDY": "IM_ACQUIRED_STUDY_ID",
        "MILL_LOCATION": "LOCATION_CD",
    }

    # If raw_cols is provided, a candidate must also exist in the raw target.
    # Staging tables get ENCNTR_ID and ORGANIZATION_ID bolted on for tagging, so
    # those columns are present in staging but often not in raw (e.g. mill_location).
    # Without this filter the MERGE ON tgt.<pk> = src.<pk> fails to resolve tgt.<pk>.
    def valid(c):
        if c not in cols_upper:
            return False
        if raw_cols_upper is not None and c not in raw_cols_upper:
            return False
        return True

    if tbl_upper in pk_map and valid(pk_map[tbl_upper]):
        return pk_map[tbl_upper]

    # Fallback: try <table_name without MILL_ prefix>_ID
    short = tbl_upper.replace("MILL_", "")
    candidate = f"{short}_ID"
    if valid(candidate):
        return candidate

    # Prefer known hub/encounter keys over an arbitrary alphabetically-first _ID.
    # mill_order_comment previously hit the alphabetical fallback and picked
    # COMMENT_PRSNL_ID (one distinct value for 1.5M rows), producing cartesian
    # joins in MERGE/DELETE and a 9000s timeout.
    for c in ("EVENT_ID", "ENCNTR_ID", "ORDER_ID", "PROBLEM_ID", "SCH_EVENT_ID",
              "SCHEDULE_ID", "EPISODE_ID", "IM_STUDY_ID", "IM_ACQUIRED_STUDY_ID",
              "SURG_CASE_ID", "DCP_FORMS_ACTIVITY_ID", "CV_PROC_ID"):
        if valid(c):
            return c

    # Last resort: first non-UPDT _ID column (previous behaviour)
    for c in sorted(cols_upper):
        if c.endswith("_ID") and c not in ("UPDT_ID", "UPDT_APPLCTX", "UPDT_CNT", "UPDT_TASK") and valid(c):
            return c

    return None


def ensure_raw_table(tbl_name, staging_fqn, raw_fqn):
    """Create raw table with schema mirroring staging (minus metadata) if it does not exist."""
    if spark.catalog.tableExists(raw_fqn):
        return False
    cols = [c for c in spark.table(staging_fqn).columns
            if c.upper() not in ("_STAGED_AT", "_CLASSIFICATION_ATTEMPTS")]
    col_list = ", ".join(f"`{c}`" for c in cols)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {raw_fqn}
        USING DELTA
        AS SELECT {col_list} FROM {staging_fqn} WHERE 1=0
    """)
    log_status(tbl_name, "INIT", f"created empty raw table {raw_fqn}")
    return True


def ensure_abandon_table(tbl_name, staging_fqn, abandon_fqn):
    """Create abandon table with full staging schema plus two bookkeeping columns."""
    if spark.catalog.tableExists(abandon_fqn):
        return False
    cols = spark.table(staging_fqn).columns
    col_list = ", ".join(f"`{c}`" for c in cols)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {abandon_fqn}
        USING DELTA
        AS SELECT {col_list},
                  CAST(NULL AS TIMESTAMP) AS _abandoned_at,
                  CAST(NULL AS STRING) AS _abandon_reason
           FROM {staging_fqn} WHERE 1=0
    """)
    log_status(tbl_name, "INIT", f"created abandon table {abandon_fqn}")
    return True


print("Helper functions loaded.")

In [0]:
# SECTION 3: Infrastructure Setup

def ensure_setup():
    """Create all required schemas and tables if they do not exist."""
    print("Ensuring infrastructure exists...")

    # Schemas
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{STAGING_SCHEMA}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TMP_SCHEMA}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{ABANDONED_SCHEMA}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LOG_CATALOG}.{LOG_SCHEMA}")

    # Organization flag table
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {FLAG_TBL} (
            organization_id LONG,
            organization_name STRING,
            flag_type STRING,
            flag_reason STRING,
            first_seen TIMESTAMP,
            last_seen TIMESTAMP,
            resolved BOOLEAN,
            resolved_at TIMESTAMP
        )
    """)

    # Control table for CDC checkpointing
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CONTROL_TBL} (
            table_name STRING,
            checkpoint_key STRING,
            last_version LONG,
            last_timestamp TIMESTAMP,
            updated_at TIMESTAMP
        )
    """)

    # Audit summary
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {AUDIT_SUMMARY_TBL} (
            run_timestamp TIMESTAMP,
            table_name STRING,
            total_rows LONG,
            classified_rows LONG,
            unclassified_rows LONG,
            bhrut_rows LONG,
            status STRING,
            notes STRING
        )
    """)

    # Org mapping history
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {ORG_HIST_TBL} (
            ENCNTR_ID LONG,
            ORGANIZATION_ID LONG,
            TRUST STRING,
            mapped_at TIMESTAMP
        )
    """)

    # Trust map - INSERT OVERWRITE with all org values
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TRUST_MAP_TBL} (
            organization_id LONG,
            trust STRING
        )
    """)

    barts_values = ", ".join([f"({o}, 'Barts')" for o in BARTS_ORGS])
    bhrut_values = ", ".join([f"({o}, 'Bhrut')" for o in BHRUT_ORGS])
    all_values = f"{barts_values}, {bhrut_values}"
    spark.sql(f"""
        INSERT OVERWRITE {TRUST_MAP_TBL}
        VALUES {all_values}
    """)
    print(f"  Trust map refreshed with {len(BARTS_ORGS)} BARTS + {len(BHRUT_ORGS)} BHRUT orgs")

    # Encounter-org map
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {ENC_ORG_TBL} (
            ENCNTR_ID LONG,
            ORGANIZATION_ID LONG,
            updated_at TIMESTAMP
        )
    """)

    # Hub table
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {HUB_TBL} (
            key_type STRING,
            key_id LONG,
            ENCNTR_ID LONG,
            updated_at TIMESTAMP
        )
    """)

    # Changed encounters tracking
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CHANGED_ENC_TBL} (
            ENCNTR_ID LONG,
            change_type STRING,
            detected_at TIMESTAMP
        )
    """)

    # Classification log table
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CLASSIFICATION_LOG_TBL} (
            run_timestamp TIMESTAMP,
            table_name STRING,
            staged_count LONG,
            classified_count LONG,
            bhrut_deleted_count LONG,
            unclassified_count LONG,
            promoted_to_raw LONG,
            error_message STRING
        )
    """)
    # Backfill error_message column on pre-existing deployments (idempotent)
    try:
        spark.sql(f"ALTER TABLE {CLASSIFICATION_LOG_TBL} ADD COLUMNS (error_message STRING)")
    except Exception:
        pass

    print("Infrastructure setup complete.")

In [0]:
# SECTION 4: CDC Utilities

def get_cdc_window(fqn, checkpoint_key):
    """
    Determine the CDC window for a table.
    Returns:
      (start_version, end_version) - normal CDF mode
      ("FALLBACK", last_timestamp)  - fallback to timestamp filtering
      (None, None)                  - no data / skip
    """
    # Get current table version
    try:
        history_df = spark.sql(f"DESCRIBE HISTORY {fqn} LIMIT 1")
        current_version = history_df.collect()[0]["version"]
    except Exception as e:
        print(f"  WARNING: Cannot read history for {fqn}: {e}")
        return (None, None)

    # Check for existing checkpoint
    try:
        ctl = spark.sql(f"""
            SELECT last_version, last_timestamp
            FROM {CONTROL_TBL}
            WHERE table_name = '{fqn}' AND checkpoint_key = '{checkpoint_key}'
        """).collect()
    except Exception:
        ctl = []

    if ctl:
        last_version = ctl[0]["last_version"]
        last_ts = ctl[0]["last_timestamp"]

        if last_version is not None and last_version < current_version:
            # Try CDF
            try:
                test_df = spark.read.format("delta") \
                    .option("readChangeFeed", "true") \
                    .option("startingVersion", last_version + 1) \
                    .option("endingVersion", current_version) \
                    .table(fqn)
                _ = test_df.limit(1).count()
                return (last_version + 1, current_version)
            except Exception as cdf_err:
                print(f"  CDF failed for {fqn} (v{last_version+1}..{current_version}): {type(cdf_err).__name__}: {str(cdf_err).splitlines()[0][:200]}")
                # Fall back to timestamp
                if last_ts is not None:
                    return ("FALLBACK", last_ts)
                else:
                    return ("FALLBACK", RUN_TS - timedelta(hours=FALLBACK_LOOKBACK_HOURS))
        elif last_version is not None and last_version >= current_version:
            # Already up to date
            return (None, None)
        else:
            # last_version is None but checkpoint exists - use timestamp fallback
            if last_ts is not None:
                return ("FALLBACK", last_ts)
            return (None, None)
    else:
        # No checkpoint: initial load with lookback
        start_ver = max(0, current_version - INITIAL_LOOKBACK_VERSIONS)
        try:
            test_df = spark.read.format("delta") \
                .option("readChangeFeed", "true") \
                .option("startingVersion", start_ver) \
                .option("endingVersion", current_version) \
                .table(fqn)
            _ = test_df.limit(1).count()
            return (start_ver, current_version)
        except Exception:
            # Fallback for initial load
            fallback_ts = RUN_TS - timedelta(hours=FALLBACK_LOOKBACK_HOURS)
            return ("FALLBACK", fallback_ts)


def update_checkpoint(fqn, version, timestamp, checkpoint_key):
    """Merge a new checkpoint entry into the control table."""
    spark.sql(f"""
        MERGE INTO {CONTROL_TBL} AS tgt
        USING (SELECT
            '{fqn}' AS table_name,
            '{checkpoint_key}' AS checkpoint_key,
            {version if version is not None else 'NULL'} AS last_version,
            TIMESTAMP '{timestamp}' AS last_timestamp,
            current_timestamp() AS updated_at
        ) AS src
        ON tgt.table_name = src.table_name AND tgt.checkpoint_key = src.checkpoint_key
        WHEN MATCHED THEN UPDATE SET
            tgt.last_version = src.last_version,
            tgt.last_timestamp = src.last_timestamp,
            tgt.updated_at = src.updated_at
        WHEN NOT MATCHED THEN INSERT *
    """)


def get_fallback_filter(fqn, last_ts):
    """Build a timestamp filter expression for fallback mode."""
    time_col = pick_time_col(fqn)
    if time_col is None:
        return None
    ts_str = last_ts.strftime("%Y-%m-%d %H:%M:%S")
    return f"{time_col} >= '{ts_str}'"


print("CDC utilities loaded.")

In [0]:
# SECTION 5: Hub Refresh

def refresh_hub():
    """
    Refresh the hub mapping tables by reading CDC changes from raw tables.
    Phase 1 of the pipeline.
    """
    print("=" * 70)
    print("PHASE 1: HUB REFRESH (CDC from raw tables)")
    print("=" * 70)

    # ------------------------------------------------------------------
    # 5a. SYNC ENCOUNTERS
    # ------------------------------------------------------------------
    print("\n--- 5a: Syncing encounters from mill_encounter ---")
    enc_fqn = f"{CATALOG}.{RAW_SCHEMA}.mill_encounter"
    enc_checkpoint_key = "hub_encounter"

    start_ver, end_ver = get_cdc_window(enc_fqn, enc_checkpoint_key)
    enc_changes = None

    if start_ver is None and end_ver is None:
        print("  mill_encounter: no new changes detected, skipping.")
    elif start_ver == "FALLBACK":
        last_ts = end_ver
        filt = get_fallback_filter(enc_fqn, last_ts)
        if filt:
            print(f"  mill_encounter: FALLBACK mode with filter: {filt}")
            enc_changes = spark.sql(f"""
                SELECT ENCNTR_ID, ORGANIZATION_ID
                FROM {enc_fqn}
                WHERE {filt}
                  AND ENCNTR_ID IS NOT NULL
                  AND ENCNTR_ID != 0
            """)
        else:
            print("  mill_encounter: no suitable time column for fallback, skipping.")
    else:
        print(f"  mill_encounter: CDF mode versions {start_ver}..{end_ver}")
        enc_changes = spark.read.format("delta") \
            .option("readChangeFeed", "true") \
            .option("startingVersion", start_ver) \
            .option("endingVersion", end_ver) \
            .table(enc_fqn) \
            .filter("_change_type != 'update_preimage'") \
            .select("ENCNTR_ID", "ORGANIZATION_ID") \
            .filter("ENCNTR_ID IS NOT NULL AND ENCNTR_ID != 0")

    if enc_changes is not None:
        enc_count = enc_changes.count()
        print(f"  Encounter changes found: {enc_count}")

        if enc_count > 0:
            enc_changes.createOrReplaceTempView("_enc_changes")

            # Merge into enc_org_map
            spark.sql(f"""
                MERGE INTO {ENC_ORG_TBL} AS tgt
                USING (
                    SELECT ENCNTR_ID, ORGANIZATION_ID
                    FROM (
                        SELECT ENCNTR_ID, ORGANIZATION_ID,
                               ROW_NUMBER() OVER (PARTITION BY ENCNTR_ID ORDER BY ORGANIZATION_ID DESC) AS rn
                        FROM _enc_changes
                    )
                    WHERE rn = 1
                ) AS src
                ON tgt.ENCNTR_ID = src.ENCNTR_ID
                WHEN MATCHED AND tgt.ORGANIZATION_ID != src.ORGANIZATION_ID THEN UPDATE SET
                    tgt.ORGANIZATION_ID = src.ORGANIZATION_ID,
                    tgt.updated_at = current_timestamp()
                WHEN NOT MATCHED THEN INSERT (ENCNTR_ID, ORGANIZATION_ID, updated_at)
                    VALUES (src.ENCNTR_ID, src.ORGANIZATION_ID, current_timestamp())
            """)

            # Track changed encounters
            spark.sql(f"""
                INSERT INTO {CHANGED_ENC_TBL}
                SELECT DISTINCT ENCNTR_ID, 'encounter_update' AS change_type, current_timestamp() AS detected_at
                FROM _enc_changes
            """)

        # Update checkpoint
        if start_ver != "FALLBACK":
            update_checkpoint(enc_fqn, end_ver, RUN_TS, enc_checkpoint_key)
        else:
            # Save current version so next run can try CDF instead of falling back again
            _cur_ver = spark.sql(f"DESCRIBE HISTORY {enc_fqn} LIMIT 1").collect()[0]["version"]
            update_checkpoint(enc_fqn, _cur_ver, RUN_TS, enc_checkpoint_key)
    else:
        print("  No encounter changes to process.")

    # ------------------------------------------------------------------
    # 5b. SYNC KEYS
    # ------------------------------------------------------------------
    print("\n--- 5b: Syncing key lookups ---")

    for key_type, source_fqn, enc_col_expr in KEY_LOOKUPS:
        checkpoint_key = f"hub_{key_type}"
        print(f"\n  Processing {key_type} from {source_fqn}...")

        s_ver, e_ver = get_cdc_window(source_fqn, checkpoint_key)
        key_changes = None

        if s_ver is None and e_ver is None:
            print(f"    {key_type}: no changes, skipping.")
            continue
        elif s_ver == "FALLBACK":
            last_ts = e_ver
            filt = get_fallback_filter(source_fqn, last_ts)
            if filt:
                print(f"    {key_type}: FALLBACK mode")
                key_changes = spark.sql(f"SELECT * FROM {source_fqn} WHERE {filt}")
            else:
                print(f"    {key_type}: no time column for fallback, skipping.")
                continue
        else:
            print(f"    {key_type}: CDF mode versions {s_ver}..{e_ver}")
            key_changes = spark.read.format("delta") \
                .option("readChangeFeed", "true") \
                .option("startingVersion", s_ver) \
                .option("endingVersion", e_ver) \
                .table(source_fqn) \
                .filter("_change_type != 'update_preimage'")

        if key_changes is None:
            continue

        key_changes.createOrReplaceTempView("_key_raw")

        # Special handling for SURG_CASE_PROC_ID: join to mill_surgical_case for ENCNTR_ID
        if key_type == "SURG_CASE_PROC_ID":
            resolved_sql = f"""
                SELECT k.SURG_CASE_PROC_ID AS key_id, sc.ENCNTR_ID
                FROM _key_raw k
                JOIN {CATALOG}.{RAW_SCHEMA}.mill_surgical_case sc ON k.SURG_CASE_ID = sc.SURG_CASE_ID
                WHERE sc.ENCNTR_ID IS NOT NULL AND sc.ENCNTR_ID != 0
            """
        elif key_type == "IM_STUDY_ID":
            # IM_STUDY_ID: also link via ORIG_ENTITY_ID to cv_proc
            resolved_sql = f"""
                SELECT k.IM_STUDY_ID AS key_id, COALESCE(k.ENCNTR_ID, cp.ENCNTR_ID) AS ENCNTR_ID
                FROM _key_raw k
                LEFT JOIN {CATALOG}.{RAW_SCHEMA}.mill_cv_proc cp ON k.ORIG_ENTITY_ID = cp.CV_PROC_ID
                WHERE COALESCE(k.ENCNTR_ID, cp.ENCNTR_ID) IS NOT NULL
                  AND COALESCE(k.ENCNTR_ID, cp.ENCNTR_ID) != 0
            """
        elif key_type == "IM_ACQUIRED_STUDY_ID":
            # Chain through mill_im_study -> mill_cv_proc
            resolved_sql = f"""
                SELECT k.IM_ACQUIRED_STUDY_ID AS key_id,
                       COALESCE(ims.ENCNTR_ID, cp.ENCNTR_ID) AS ENCNTR_ID
                FROM _key_raw k
                JOIN {CATALOG}.{RAW_SCHEMA}.mill_im_study ims ON k.MATCHED_STUDY_ID = ims.IM_STUDY_ID
                LEFT JOIN {CATALOG}.{RAW_SCHEMA}.mill_cv_proc cp ON ims.ORIG_ENTITY_ID = cp.CV_PROC_ID
                WHERE COALESCE(ims.ENCNTR_ID, cp.ENCNTR_ID) IS NOT NULL
                  AND COALESCE(ims.ENCNTR_ID, cp.ENCNTR_ID) != 0
            """
        else:
            # Standard: enc_col_expr is the column/expression for ENCNTR_ID
            resolved_sql = f"""
                SELECT {key_type} AS key_id, {enc_col_expr} AS ENCNTR_ID
                FROM _key_raw
                WHERE {enc_col_expr} IS NOT NULL AND {enc_col_expr} != 0
            """

        resolved_df = spark.sql(resolved_sql)
        resolved_df.createOrReplaceTempView("_key_resolved")

        merge_count = resolved_df.count()
        print(f"    {key_type}: {merge_count} resolved keys")

        if merge_count > 0:
            spark.sql(f"""
                MERGE INTO {HUB_TBL} AS tgt
                USING (
                    SELECT '{key_type}' AS key_type, key_id, ENCNTR_ID
                    FROM (
                        SELECT key_id, ENCNTR_ID,
                               ROW_NUMBER() OVER (PARTITION BY key_id ORDER BY ENCNTR_ID DESC) AS rn
                        FROM _key_resolved
                    )
                    WHERE rn = 1
                ) AS src
                ON tgt.key_type = src.key_type AND tgt.key_id = src.key_id
                WHEN MATCHED AND tgt.ENCNTR_ID != src.ENCNTR_ID THEN UPDATE SET
                    tgt.ENCNTR_ID = src.ENCNTR_ID,
                    tgt.updated_at = current_timestamp()
                WHEN NOT MATCHED THEN INSERT (key_type, key_id, ENCNTR_ID, updated_at)
                    VALUES (src.key_type, src.key_id, src.ENCNTR_ID, current_timestamp())
            """)

        # Update checkpoint
        if s_ver != "FALLBACK":
            update_checkpoint(source_fqn, e_ver, RUN_TS, checkpoint_key)
        else:
            _cur_ver = spark.sql(f"DESCRIBE HISTORY {source_fqn} LIMIT 1").collect()[0]["version"]
            update_checkpoint(source_fqn, _cur_ver, RUN_TS, checkpoint_key)

    # ------------------------------------------------------------------
    # 5c. CLINICAL EVENTS
    # ------------------------------------------------------------------
    print("\n--- 5c: Syncing clinical events ---")
    ce_fqn = f"{CATALOG}.{RAW_SCHEMA}.mill_clinical_event"
    ce_checkpoint_key = "hub_clinical_event"

    ce_s, ce_e = get_cdc_window(ce_fqn, ce_checkpoint_key)
    ce_changes = None

    if ce_s is None and ce_e is None:
        print("  mill_clinical_event: no changes, skipping.")
    elif ce_s == "FALLBACK":
        last_ts = ce_e
        filt = get_fallback_filter(ce_fqn, last_ts)
        if filt:
            print(f"  mill_clinical_event: FALLBACK mode")
            ce_changes = spark.sql(f"""
                SELECT EVENT_ID, ENCNTR_ID
                FROM {ce_fqn}
                WHERE {filt}
                  AND EVENT_ID IS NOT NULL AND EVENT_ID != 0
                  AND ENCNTR_ID IS NOT NULL AND ENCNTR_ID != 0
            """)
        else:
            print("  mill_clinical_event: no time column for fallback.")
    else:
        print(f"  mill_clinical_event: CDF mode versions {ce_s}..{ce_e}")
        ce_changes = spark.read.format("delta") \
            .option("readChangeFeed", "true") \
            .option("startingVersion", ce_s) \
            .option("endingVersion", ce_e) \
            .table(ce_fqn) \
            .filter("_change_type != 'update_preimage'") \
            .select("EVENT_ID", "ENCNTR_ID") \
            .filter("EVENT_ID IS NOT NULL AND EVENT_ID != 0 AND ENCNTR_ID IS NOT NULL AND ENCNTR_ID != 0")

    if ce_changes is not None:
        ce_count = ce_changes.count()
        print(f"  Clinical event changes: {ce_count}")

        if ce_count > 0:
            ce_changes.createOrReplaceTempView("_ce_changes")

            spark.sql(f"""
                MERGE INTO {HUB_TBL} AS tgt
                USING (
                    SELECT 'EVENT_ID' AS key_type, EVENT_ID AS key_id, ENCNTR_ID
                    FROM (
                        SELECT EVENT_ID, ENCNTR_ID,
                               ROW_NUMBER() OVER (PARTITION BY EVENT_ID ORDER BY ENCNTR_ID DESC) AS rn
                        FROM _ce_changes
                    )
                    WHERE rn = 1
                ) AS src
                ON tgt.key_type = src.key_type AND tgt.key_id = src.key_id
                WHEN MATCHED AND tgt.ENCNTR_ID != src.ENCNTR_ID THEN UPDATE SET
                    tgt.ENCNTR_ID = src.ENCNTR_ID,
                    tgt.updated_at = current_timestamp()
                WHEN NOT MATCHED THEN INSERT (key_type, key_id, ENCNTR_ID, updated_at)
                    VALUES (src.key_type, src.key_id, src.ENCNTR_ID, current_timestamp())
            """)

        if ce_s != "FALLBACK":
            update_checkpoint(ce_fqn, ce_e, RUN_TS, ce_checkpoint_key)
        else:
            _cur_ver = spark.sql(f"DESCRIBE HISTORY {ce_fqn} LIMIT 1").collect()[0]["version"]
            update_checkpoint(ce_fqn, _cur_ver, RUN_TS, ce_checkpoint_key)

    # ------------------------------------------------------------------
    # 5d. CLEANUP old entries
    # ------------------------------------------------------------------
    print("\n--- 5d: Cleaning up old hub entries ---")
    cutoff_ts = (RUN_TS - timedelta(days=LOOKUP_RETENTION_DAYS)).strftime("%Y-%m-%d %H:%M:%S")

    hub_deleted = spark.sql(f"""
        DELETE FROM {HUB_TBL} WHERE updated_at < '{cutoff_ts}'
    """)
    print(f"  Cleaned hub entries older than {LOOKUP_RETENTION_DAYS} days")

    enc_deleted = spark.sql(f"""
        DELETE FROM {ENC_ORG_TBL} WHERE updated_at < '{cutoff_ts}'
    """)
    print(f"  Cleaned enc_org entries older than {LOOKUP_RETENTION_DAYS} days")

    # Clear changed encounters for next run
    spark.sql(f"TRUNCATE TABLE {CHANGED_ENC_TBL}")

    print("\nHub refresh complete.")

In [0]:
# SECTION 6: Materialize Hub Views

def materialize_hub_views():
    """
    Materialize lookup data as REAL Delta tables (not lazy temp views).

    Why: the resolution SQL references enc_trust_lookup twice per table
    (once as etl_direct, once as et0 through the hub). Lazy views re-evaluate
    on each reference. Real tables give the optimizer statistics for
    stable BROADCAST decisions and let us ZORDER the hub for key lookups.
    Temp-view aliases are kept so existing resolution SQL reads unchanged.
    """
    print("Materializing hub lookup tables...")

    enc_org_fqn = f"{CATALOG}.{TMP_SCHEMA}._enc_org_deduped"
    hub_fqn = f"{CATALOG}.{TMP_SCHEMA}._hub_deduped"
    trust_fqn = f"{CATALOG}.{TMP_SCHEMA}._trust_map_view"
    enc_trust_fqn = f"{CATALOG}.{TMP_SCHEMA}._enc_trust_lookup"

    spark.sql(f"""
        CREATE OR REPLACE TABLE {enc_org_fqn}
        USING DELTA
        AS SELECT ENCNTR_ID, ORGANIZATION_ID FROM {ENC_ORG_TBL}
    """)

    spark.sql(f"""
        CREATE OR REPLACE TABLE {hub_fqn}
        USING DELTA
        AS SELECT key_type, key_id, ENCNTR_ID FROM {HUB_TBL}
    """)

    spark.sql(f"""
        CREATE OR REPLACE TABLE {trust_fqn}
        USING DELTA
        AS SELECT organization_id, trust FROM {TRUST_MAP_TBL}
    """)

    # Pre-joined encounter -> trust lookup. Materialising this is the single
    # biggest win since the resolution SQL references it twice.
    spark.sql(f"""
        CREATE OR REPLACE TABLE {enc_trust_fqn}
        USING DELTA
        AS
        SELECT /*+ BROADCAST(tm) */
               eo.ENCNTR_ID, eo.ORGANIZATION_ID, tm.trust
        FROM {enc_org_fqn} eo
        JOIN {trust_fqn} tm ON eo.ORGANIZATION_ID = tm.organization_id
    """)

    for fqn, zcols in [(hub_fqn, "key_type, key_id"), (enc_trust_fqn, "ENCNTR_ID")]:
        try:
            spark.sql(f"OPTIMIZE {fqn} ZORDER BY ({zcols})")
        except Exception as e:
            print(f"  OPTIMIZE {fqn} warning: {type(e).__name__}: {str(e).splitlines()[0][:200]}")

    # Expose the same names as temp views so the rest of the pipeline reads
    # unchanged. These are pass-throughs over the Delta tables above, so stats
    # and data-skipping still apply.
    spark.sql(f"CREATE OR REPLACE TEMPORARY VIEW enc_org_deduped  AS SELECT * FROM {enc_org_fqn}")
    spark.sql(f"CREATE OR REPLACE TEMPORARY VIEW hub_deduped      AS SELECT * FROM {hub_fqn}")
    spark.sql(f"CREATE OR REPLACE TEMPORARY VIEW trust_map        AS SELECT * FROM {trust_fqn}")
    spark.sql(f"CREATE OR REPLACE TEMPORARY VIEW enc_trust_lookup AS SELECT * FROM {enc_trust_fqn}")

    enc_org_cnt   = spark.sql(f"SELECT COUNT(*) AS c FROM {enc_org_fqn}").collect()[0]["c"]
    hub_cnt       = spark.sql(f"SELECT COUNT(*) AS c FROM {hub_fqn}").collect()[0]["c"]
    enc_trust_cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {enc_trust_fqn}").collect()[0]["c"]
    print(f"  enc_org_deduped:  {enc_org_cnt}")
    print(f"  hub_deduped:      {hub_cnt}")
    print(f"  enc_trust_lookup: {enc_trust_cnt}")
    print("Hub lookup tables materialized.")

In [0]:
# SECTION 7: Staging Discovery

def discover_staging_tables():
    """
    Discover all staging tables that have rows waiting for classification.
    Returns dict: table_name -> {staging_fqn, raw_fqn, cols, row_count}
    """
    print("Discovering staging tables...")

    tables_df = spark.sql(f"SHOW TABLES IN {CATALOG}.{STAGING_SCHEMA}").collect()
    candidates = {}

    for row in tables_df:
        tbl_name = row["tableName"]
        if not tbl_name.startswith("mill_"):
            print(f"  Skipping non-mill staging table: {tbl_name}")
            continue
        staging_fqn = f"{CATALOG}.{STAGING_SCHEMA}.{tbl_name}"
        raw_fqn = f"{CATALOG}.{RAW_SCHEMA}.{tbl_name}"

        try:
            cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {staging_fqn}").collect()[0]["c"]
        except Exception:
            continue

        if cnt == 0:
            continue

        # Get columns, excluding staging-specific metadata
        try:
            all_cols = [c.upper() for c in spark.table(staging_fqn).columns]
            cols = [c for c in all_cols if c not in ("_STAGED_AT", "_CLASSIFICATION_ATTEMPTS")]
        except Exception:
            continue

        candidates[tbl_name] = {
            "staging_fqn": staging_fqn,
            "raw_fqn": raw_fqn,
            "cols": cols,
            "row_count": cnt
        }
        log_status(tbl_name, "QUEUE", f"{cnt} rows in staging")

    print(f"Found {len(candidates)} staging tables with pending rows.")
    return candidates

In [0]:
# SECTION 8: Trust Resolution SQL Builder

def build_staging_trust_resolution_sql(table_name, cols, pk_col, staging_fqn):
    """
    Build SQL that resolves trust for all staging rows.
    Returns SQL producing: (pk_col, new_enc_id, new_org_id, new_trust)
    new_trust is NULL for rows that cannot be resolved.
    """
    tbl_upper = table_name.upper()
    cols_upper = {c.upper() for c in cols}

    # Build join clauses for trust resolution - try multiple paths
    # Path 1: Direct ORGANIZATION_ID -> trust_map
    # Path 2: ENCNTR_ID -> enc_trust_lookup
    # Path 3: Hub key columns -> hub_deduped -> enc_trust_lookup

    # Identify hub key columns present in this table
    hub_key_cols = []
    for key_type, source_fqn, enc_expr in KEY_LOOKUPS:
        if key_type in cols_upper:
            hub_key_cols.append(key_type)

    # Also check EVENT_ID for clinical event tables
    if "EVENT_ID" in cols_upper:
        hub_key_cols.append("EVENT_ID")

    # Build the resolution SQL
    hub_joins = ""
    hub_coalesce_enc = ""
    hub_coalesce_org = ""
    hub_coalesce_trust = ""

    for i, key_col in enumerate(hub_key_cols):
        alias_h = f"h{i}"
        alias_e = f"et{i}"
        hub_joins += f"""
        LEFT JOIN hub_deduped {alias_h}
            ON {alias_h}.key_type = '{key_col}' AND {alias_h}.key_id = s.{key_col}
        LEFT JOIN enc_trust_lookup {alias_e}
            ON {alias_e}.ENCNTR_ID = {alias_h}.ENCNTR_ID
        """
        hub_coalesce_enc += f", {alias_e}.ENCNTR_ID"
        hub_coalesce_org += f", {alias_e}.ORGANIZATION_ID"
        hub_coalesce_trust += f", {alias_e}.trust"

    # Also handle chained dependencies
    for key_col, (chain_tbl, chain_fk, chain_pk) in CHAINED_DEPENDENCIES.items():
        if chain_pk in cols_upper and chain_pk not in hub_key_cols:
            ci = len(hub_key_cols) + list(CHAINED_DEPENDENCIES.keys()).index(key_col)
            alias_h = f"h{ci}"
            alias_e = f"et{ci}"
            hub_joins += f"""
            LEFT JOIN hub_deduped {alias_h}
                ON {alias_h}.key_type = '{key_col}' AND {alias_h}.key_id = s.{chain_pk}
            LEFT JOIN enc_trust_lookup {alias_e}
                ON {alias_e}.ENCNTR_ID = {alias_h}.ENCNTR_ID
            """
            hub_coalesce_enc += f", {alias_e}.ENCNTR_ID"
            hub_coalesce_org += f", {alias_e}.ORGANIZATION_ID"
            hub_coalesce_trust += f", {alias_e}.trust"

    # Build the full SQL
    has_org = "ORGANIZATION_ID" in cols_upper
    has_enc = "ENCNTR_ID" in cols_upper

    enc_coalesce_parts = []
    org_coalesce_parts = []
    trust_coalesce_parts = []

    # Path 1: Direct org
    if has_org:
        org_coalesce_parts.append("tm_direct.organization_id")
        trust_coalesce_parts.append("tm_direct.trust")

    # Path 2: Direct encounter
    if has_enc:
        enc_coalesce_parts.append("etl_direct.ENCNTR_ID")
        org_coalesce_parts.append("etl_direct.ORGANIZATION_ID")
        trust_coalesce_parts.append("etl_direct.trust")

    # Path 3: Hub keys (already built above)
    if hub_coalesce_enc:
        enc_coalesce_parts.append(hub_coalesce_enc.lstrip(", "))
        org_coalesce_parts.append(hub_coalesce_org.lstrip(", "))
        trust_coalesce_parts.append(hub_coalesce_trust.lstrip(", "))

    # Build COALESCE expressions. Typed CAST(NULL AS ...) matters when no
    # resolution path applies: materialising a bare NULL column to Delta
    # produces a VOID type which fails on re-read in the COUNT query below.
    new_enc_expr = f"COALESCE({', '.join(enc_coalesce_parts)})" if enc_coalesce_parts else "CAST(NULL AS BIGINT)"
    if has_enc:
        new_enc_expr = f"COALESCE(s.ENCNTR_ID, {new_enc_expr})" if enc_coalesce_parts else "s.ENCNTR_ID"

    new_org_expr = f"COALESCE({', '.join(org_coalesce_parts)})" if org_coalesce_parts else "CAST(NULL AS BIGINT)"
    if has_org:
        new_org_expr = f"COALESCE(s.ORGANIZATION_ID, {new_org_expr})" if org_coalesce_parts else "s.ORGANIZATION_ID"

    new_trust_expr = f"COALESCE({', '.join(trust_coalesce_parts)})" if trust_coalesce_parts else "CAST(NULL AS STRING)"

    # Build join clauses for direct paths
    direct_joins = ""
    if has_org:
        direct_joins += """
        LEFT JOIN trust_map tm_direct ON tm_direct.organization_id = s.ORGANIZATION_ID
        """
    if has_enc:
        direct_joins += """
        LEFT JOIN enc_trust_lookup etl_direct ON etl_direct.ENCNTR_ID = s.ENCNTR_ID
        """

    # BROADCAST hint for small lookup tables
    broadcast_tables = []
    if has_org:
        broadcast_tables.append("tm_direct")  # trust_map: ~47 rows
    hint_sql = f"/*+ BROADCAST({', '.join(broadcast_tables)}) */ " if broadcast_tables else ""

    sql = f"""
    SELECT {hint_sql}
        s.{pk_col} AS pk_val,
        {new_enc_expr} AS new_enc_id,
        {new_org_expr} AS new_org_id,
        {new_trust_expr} AS new_trust
    FROM {staging_fqn} s
    {direct_joins}
    {hub_joins}
    """

    return sql

In [0]:
# SECTION 9: Classify and Promote

def classify_and_promote(staging_tables):
    """
    For each staging table: resolve trust, promote classified rows to raw,
    handle BHRUT, increment attempts on unclassified.
    """
    print("=" * 70)
    print("PHASE 2: CLASSIFY STAGING & PROMOTE TO RAW")
    print("=" * 70)

    for tbl_name, info in staging_tables.items():
        staging_fqn = info["staging_fqn"]
        raw_fqn = info["raw_fqn"]
        cols = info["cols"]
        row_count = info["row_count"]
        tbl_upper = tbl_name.upper()

        log_status(tbl_name, "PROC", f"classifying {row_count} staged rows")

        resolved_fqn = f"{CATALOG}.{TMP_SCHEMA}._resolved_{tbl_name}"

        try:
            # Ensure raw target table exists (create empty shell matching staging schema if missing)
            ensure_raw_table(tbl_name, staging_fqn, raw_fqn)

            # Determine PK. Filter candidates against raw_cols so we never pick
            # a column that only exists in staging.
            try:
                raw_cols_list = spark.table(raw_fqn).columns
            except Exception:
                raw_cols_list = None
            pk_col = determine_pk_column(tbl_name, cols, raw_cols=raw_cols_list)
            if pk_col is None:
                log_status(tbl_name, "ERR", "cannot determine PK column")
                continue

            # Build trust resolution SQL and materialize as a Delta temp table
            # so the resolution computes ONCE instead of re-running for every downstream
            # count / MERGE / DELETE / UPDATE reference.
            resolution_sql = build_staging_trust_resolution_sql(tbl_name, cols, pk_col, staging_fqn)
            spark.sql(f"DROP TABLE IF EXISTS {resolved_fqn}")
            # Pre-aggregate by pk_val so downstream JOINs/IN-subqueries see at most
            # one row per PK.
            spark.sql(f"""
                CREATE TABLE {resolved_fqn}
                USING DELTA
                AS
                SELECT pk_val,
                       MAX(new_enc_id) AS new_enc_id,
                       MAX(new_org_id) AS new_org_id,
                       MAX(new_trust)  AS new_trust
                FROM ({resolution_sql})
                GROUP BY pk_val
            """)

            # Count categories
            counts = spark.sql(f"""
                SELECT
                    COUNT(*) AS total,
                    SUM(CASE WHEN new_trust IS NOT NULL AND new_trust != 'Bhrut' THEN 1 ELSE 0 END) AS classified,
                    SUM(CASE WHEN new_trust = 'Bhrut' THEN 1 ELSE 0 END) AS bhrut,
                    SUM(CASE WHEN new_trust IS NULL THEN 1 ELSE 0 END) AS unclassified
                FROM {resolved_fqn}
            """).collect()[0]

            total = counts["total"]
            classified_count = counts["classified"]
            bhrut_count = counts["bhrut"]
            unclassified_count = counts["unclassified"]

            print(f"    Total: {total} | Classified: {classified_count} | BHRUT: {bhrut_count} | Unclassified: {unclassified_count}")

            promoted_count = 0

            # ---- PROMOTE CLASSIFIED rows ----
            if classified_count > 0:
                cols_upper = {c.upper() for c in cols}
                is_source_of_truth = tbl_upper in SOURCE_OF_TRUTH_TABLES

                # Build enriched source: staging columns with trust filled in
                select_parts = []
                for c in cols:
                    c_upper = c.upper()
                    if c_upper == "TRUST":
                        select_parts.append("r.new_trust AS Trust")
                    elif c_upper == "ENCNTR_ID" and not is_source_of_truth:
                        # Fill in ENCNTR_ID from resolution if originally null
                        select_parts.append("COALESCE(s.ENCNTR_ID, r.new_enc_id) AS ENCNTR_ID")
                    elif c_upper == "ORGANIZATION_ID" and not is_source_of_truth:
                        select_parts.append("COALESCE(s.ORGANIZATION_ID, r.new_org_id) AS ORGANIZATION_ID")
                    elif c_upper == "ADC_UPDT":
                        select_parts.append("current_timestamp() AS ADC_UPDT")
                    else:
                        select_parts.append(f"s.{c}")

                # Add Trust column if not already in cols
                if "TRUST" not in cols_upper:
                    select_parts.append("r.new_trust AS Trust")

                select_clause = ", ".join(select_parts)

                # Build with ROW_NUMBER dedup on pk_col
                enriched_sql = f"""
                SELECT {select_clause},
                       ROW_NUMBER() OVER (PARTITION BY s.{pk_col} ORDER BY s.{pk_col}) AS _rn
                FROM {staging_fqn} s
                JOIN {resolved_fqn} r
                    ON s.{pk_col} = r.pk_val
                WHERE r.new_trust IS NOT NULL AND r.new_trust != 'Bhrut'
                """

                spark.sql(enriched_sql).createOrReplaceTempView(f"_enriched_{tbl_name}")

                # Determine columns for merge (exclude staging metadata and _rn)
                merge_cols = list(cols)
                if "TRUST" not in [c.upper() for c in merge_cols]:
                    merge_cols.append("Trust")

                # Drop staging metadata columns and _rn for the merge source
                drop_cols = {"_STAGED_AT", "_CLASSIFICATION_ATTEMPTS", "_RN"}
                merge_col_list = [c for c in merge_cols if c.upper() not in drop_cols]

                merge_select = ", ".join(merge_col_list)

                spark.sql(f"""
                    CREATE OR REPLACE TEMPORARY VIEW _merge_source_{tbl_name} AS
                    SELECT {merge_select}
                    FROM _enriched_{tbl_name}
                    WHERE _rn = 1
                """)

                # MERGE INTO raw
                spark.sql(f"""
                    MERGE INTO {raw_fqn} AS tgt
                    USING _merge_source_{tbl_name} AS src
                    ON tgt.{pk_col} = src.{pk_col}
                    WHEN MATCHED THEN UPDATE SET *
                    WHEN NOT MATCHED THEN INSERT *
                """)

                promoted_count = classified_count

                # DELETE promoted rows from staging
                spark.sql(f"""
                    DELETE FROM {staging_fqn}
                    WHERE {pk_col} IN (
                        SELECT pk_val FROM {resolved_fqn}
                        WHERE new_trust IS NOT NULL AND new_trust != 'Bhrut'
                    )
                """)

                log_status(tbl_name, "PROC", f"promoted {promoted_count} rows to raw")

            # ---- HANDLE BHRUT rows ----
            bhrut_deleted = 0
            if bhrut_count > 0:
                if can_delete_bhrut(tbl_upper):
                    # Delete BHRUT rows from staging (discard them)
                    spark.sql(f"""
                        DELETE FROM {staging_fqn}
                        WHERE {pk_col} IN (
                            SELECT pk_val FROM {resolved_fqn}
                            WHERE new_trust = 'Bhrut'
                        )
                    """)
                    bhrut_deleted = bhrut_count
                    log_status(tbl_name, "PROC", f"deleted {bhrut_count} BHRUT rows from staging")
                else:
                    # Protected table: promote BHRUT rows WITH Trust='Bhrut'
                    cols_upper = {c.upper() for c in cols}
                    is_source_of_truth = tbl_upper in SOURCE_OF_TRUTH_TABLES

                    bhrut_select_parts = []
                    for c in cols:
                        c_upper = c.upper()
                        if c_upper == "TRUST":
                            bhrut_select_parts.append("'Bhrut' AS Trust")
                        elif c_upper == "ENCNTR_ID" and not is_source_of_truth:
                            bhrut_select_parts.append("COALESCE(s.ENCNTR_ID, r.new_enc_id) AS ENCNTR_ID")
                        elif c_upper == "ORGANIZATION_ID" and not is_source_of_truth:
                            bhrut_select_parts.append("COALESCE(s.ORGANIZATION_ID, r.new_org_id) AS ORGANIZATION_ID")
                        elif c_upper == "ADC_UPDT":
                            bhrut_select_parts.append("current_timestamp() AS ADC_UPDT")
                        else:
                            bhrut_select_parts.append(f"s.{c}")

                    if "TRUST" not in cols_upper:
                        bhrut_select_parts.append("'Bhrut' AS Trust")

                    bhrut_select = ", ".join(bhrut_select_parts)

                    bhrut_enriched_sql = f"""
                    SELECT {bhrut_select},
                           ROW_NUMBER() OVER (PARTITION BY s.{pk_col} ORDER BY s.{pk_col}) AS _rn
                    FROM {staging_fqn} s
                    JOIN {resolved_fqn} r
                        ON s.{pk_col} = r.pk_val
                    WHERE r.new_trust = 'Bhrut'
                    """

                    spark.sql(bhrut_enriched_sql).createOrReplaceTempView(f"_bhrut_{tbl_name}")

                    bhrut_merge_cols = list(cols)
                    if "TRUST" not in [c.upper() for c in bhrut_merge_cols]:
                        bhrut_merge_cols.append("Trust")
                    bhrut_merge_cols = [c for c in bhrut_merge_cols if c.upper() not in {"_STAGED_AT", "_CLASSIFICATION_ATTEMPTS", "_RN"}]
                    bhrut_merge_select = ", ".join(bhrut_merge_cols)

                    spark.sql(f"""
                        CREATE OR REPLACE TEMPORARY VIEW _bhrut_merge_{tbl_name} AS
                        SELECT {bhrut_merge_select}
                        FROM _bhrut_{tbl_name}
                        WHERE _rn = 1
                    """)

                    spark.sql(f"""
                        MERGE INTO {raw_fqn} AS tgt
                        USING _bhrut_merge_{tbl_name} AS src
                        ON tgt.{pk_col} = src.{pk_col}
                        WHEN MATCHED THEN UPDATE SET *
                        WHEN NOT MATCHED THEN INSERT *
                    """)

                    # Delete from staging after promoting
                    spark.sql(f"""
                        DELETE FROM {staging_fqn}
                        WHERE {pk_col} IN (
                            SELECT pk_val FROM {resolved_fqn}
                            WHERE new_trust = 'Bhrut'
                        )
                    """)
                    promoted_count += bhrut_count
                    log_status(tbl_name, "PROC", f"promoted {bhrut_count} BHRUT rows to raw (protected table)")

            # ---- UNCLASSIFIED: archive at cap, else increment attempts ----
            abandoned_count = 0
            if unclassified_count > 0:
                abandon_fqn = f"{CATALOG}.{ABANDONED_SCHEMA}.{tbl_name}"
                ensure_abandon_table(tbl_name, staging_fqn, abandon_fqn)

                # Rows whose current attempts value is >= MAX - 1 would exceed the
                # cap on this run. Move them to the abandon table instead of
                # incrementing, so they stop consuming compute on future runs.
                abandon_threshold = MAX_CLASSIFICATION_ATTEMPTS - 1

                if unclassified_count == total:
                    # Fast path: every staging row is unclassified. We can use a
                    # simple WHERE on the attempts counter without joining back
                    # to the resolved table.
                    abandoned_count = spark.sql(f"""
                        SELECT COUNT(*) AS c FROM {staging_fqn}
                        WHERE _classification_attempts >= {abandon_threshold}
                    """).collect()[0]["c"]
                    if abandoned_count > 0:
                        spark.sql(f"""
                            INSERT INTO {abandon_fqn}
                            SELECT *,
                                   current_timestamp() AS _abandoned_at,
                                   'max_attempts_reached' AS _abandon_reason
                            FROM {staging_fqn}
                            WHERE _classification_attempts >= {abandon_threshold}
                        """)
                        spark.sql(f"""
                            DELETE FROM {staging_fqn}
                            WHERE _classification_attempts >= {abandon_threshold}
                        """)
                    spark.sql(f"""
                        UPDATE {staging_fqn}
                        SET _classification_attempts = _classification_attempts + 1
                    """)
                else:
                    # Standard path: scope abandon + increment via resolved PKs.
                    abandoned_count = spark.sql(f"""
                        SELECT COUNT(*) AS c FROM {staging_fqn} s
                        JOIN {resolved_fqn} r ON s.{pk_col} = r.pk_val
                        WHERE r.new_trust IS NULL
                          AND s._classification_attempts >= {abandon_threshold}
                    """).collect()[0]["c"]
                    if abandoned_count > 0:
                        spark.sql(f"""
                            INSERT INTO {abandon_fqn}
                            SELECT s.*,
                                   current_timestamp() AS _abandoned_at,
                                   'max_attempts_reached' AS _abandon_reason
                            FROM {staging_fqn} s
                            JOIN {resolved_fqn} r ON s.{pk_col} = r.pk_val
                            WHERE r.new_trust IS NULL
                              AND s._classification_attempts >= {abandon_threshold}
                        """)
                        spark.sql(f"""
                            DELETE FROM {staging_fqn}
                            WHERE {pk_col} IN (
                                SELECT s.{pk_col}
                                FROM {staging_fqn} s
                                JOIN {resolved_fqn} r ON s.{pk_col} = r.pk_val
                                WHERE r.new_trust IS NULL
                                  AND s._classification_attempts >= {abandon_threshold}
                            )
                        """)
                    spark.sql(f"""
                        UPDATE {staging_fqn}
                        SET _classification_attempts = _classification_attempts + 1
                        WHERE {pk_col} IN (
                            SELECT pk_val FROM {resolved_fqn}
                            WHERE new_trust IS NULL
                        )
                    """)

                remaining = unclassified_count - abandoned_count
                if abandoned_count > 0:
                    log_status(tbl_name, "INFO",
                               f"abandoned {abandoned_count} rows at attempts>={MAX_CLASSIFICATION_ATTEMPTS} -> {abandon_fqn}")
                log_status(tbl_name, "SKIP",
                           f"{remaining} rows remain unclassified (attempts incremented)")

            # ---- LOG to classification log ----
            spark.sql(f"""
                INSERT INTO {CLASSIFICATION_LOG_TBL}
                VALUES (
                    current_timestamp(),
                    '{tbl_name}',
                    {row_count},
                    {classified_count},
                    {bhrut_deleted},
                    {unclassified_count},
                    {promoted_count},
                    NULL
                )
            """)

        except Exception as e:
            err_msg = str(e)[:2000].replace("'", "''")
            concise = f"{type(e).__name__}: {str(e).splitlines()[0][:200]}"
            log_status(tbl_name, "ERR", f"failed: {concise}")
            # Full exception text goes to the log table's error_message column below;
            # no stack trace printed to keep the driver output scannable.
            try:
                spark.sql(f"""
                    INSERT INTO {CLASSIFICATION_LOG_TBL}
                    VALUES (
                        current_timestamp(),
                        '{tbl_name}',
                        {row_count},
                        0, 0, 0, 0,
                        '{err_msg}'
                    )
                """)
            except Exception:
                pass
        finally:
            # Clean up the materialized resolution temp table so tmp schema doesn't
            # accumulate one stale _resolved_* table per source table per run.
            try:
                spark.sql(f"DROP TABLE IF EXISTS {resolved_fqn}")
            except Exception:
                pass

    print("\nClassification and promotion complete.")

In [0]:
# SECTION 10: Flag Unknown Organizations

def flag_unknown_organizations():
    """
    Identify organizations in enc_org_map that are not in the trust_map
    and flag them for investigation.
    """
    print("=" * 70)
    print("PHASE 3: FLAG UNKNOWN ORGANIZATIONS")
    print("=" * 70)

    unknown_orgs = spark.sql(f"""
        SELECT DISTINCT eo.ORGANIZATION_ID
        FROM {ENC_ORG_TBL} eo
        LEFT JOIN {TRUST_MAP_TBL} tm ON eo.ORGANIZATION_ID = tm.organization_id
        WHERE tm.organization_id IS NULL
          AND eo.ORGANIZATION_ID IS NOT NULL
          AND eo.ORGANIZATION_ID != 0
    """)

    unknown_count = unknown_orgs.count()

    if unknown_count == 0:
        print("  No unknown organizations found.")
        return

    print(f"  Found {unknown_count} unknown organizations.")
    unknown_orgs.createOrReplaceTempView("_unknown_orgs")

    # Try to get organization names
    try:
        org_details = spark.sql(f"""
            SELECT u.ORGANIZATION_ID,
                   COALESCE(o.ORG_NAME, 'UNKNOWN') AS organization_name
            FROM _unknown_orgs u
            LEFT JOIN {CATALOG}.{RAW_SCHEMA}.mill_organization o
                ON u.ORGANIZATION_ID = o.ORGANIZATION_ID
        """)
        org_details.createOrReplaceTempView("_unknown_org_details")
    except Exception:
        spark.sql("""
            SELECT ORGANIZATION_ID, 'UNKNOWN' AS organization_name
            FROM _unknown_orgs
        """).createOrReplaceTempView("_unknown_org_details")

    # Merge flags
    spark.sql(f"""
        MERGE INTO {FLAG_TBL} AS tgt
        USING (
            SELECT
                ORGANIZATION_ID AS organization_id,
                organization_name,
                'UNKNOWN_TRUST' AS flag_type,
                'Organization not mapped to any trust in org_to_trust_map' AS flag_reason,
                current_timestamp() AS first_seen,
                current_timestamp() AS last_seen,
                false AS resolved,
                CAST(NULL AS TIMESTAMP) AS resolved_at
            FROM _unknown_org_details
        ) AS src
        ON tgt.organization_id = src.organization_id AND tgt.flag_type = src.flag_type AND tgt.resolved = false
        WHEN MATCHED THEN UPDATE SET
            tgt.last_seen = current_timestamp(),
            tgt.organization_name = src.organization_name
        WHEN NOT MATCHED THEN INSERT *
    """)

    # Resolve flags for orgs that are now known
    spark.sql(f"""
        UPDATE {FLAG_TBL}
        SET resolved = true, resolved_at = current_timestamp()
        WHERE flag_type = 'UNKNOWN_TRUST'
          AND resolved = false
          AND organization_id IN (SELECT organization_id FROM {TRUST_MAP_TBL})
    """)

    # Show current unresolved flags
    unresolved = spark.sql(f"""
        SELECT organization_id, organization_name, flag_reason, first_seen, last_seen
        FROM {FLAG_TBL}
        WHERE flag_type = 'UNKNOWN_TRUST' AND resolved = false
        ORDER BY last_seen DESC
    """)
    print(f"  Current unresolved unknown org flags: {unresolved.count()}")
    if unresolved.count() > 0 and unresolved.count() <= 50:
        unresolved.show(50, truncate=False)

    print("Unknown organization flagging complete.")

In [0]:
# SECTION 11: Main Execution

if __name__ == "__main__":
    pipeline_start = time.time()
    print("=" * 70)
    print("TRUST CLASSIFICATION INCREMENTAL PIPELINE")
    print(f"TARGET_ENV: {TARGET_ENV} | CATALOG: {CATALOG}")
    print(f"Run started at: {RUN_TS}")
    print("=" * 70)

    # Step 1: Setup infrastructure
    time_op("Infrastructure Setup", ensure_setup)

    # Step 2: Hub refresh (Phase 1 - CDC from raw tables)
    time_op("Hub Refresh", refresh_hub)

    # Step 3: Materialize hub views for classification
    time_op("Materialize Hub Views", materialize_hub_views)

    # Step 4: Discover staging tables
    staging_tables = time_op("Discover Staging Tables", discover_staging_tables)

    # Step 5: Classify and promote (Phase 2)
    if staging_tables:
        time_op("Classify & Promote", lambda: classify_and_promote(staging_tables))
    else:
        print("\nNo staging tables with pending rows found. Skipping classification.")

    # Step 6: Flag unknown organizations (Phase 3)
    time_op("Flag Unknown Orgs", flag_unknown_organizations)

    # Summary
    pipeline_elapsed = time.time() - pipeline_start
    print("\n" + "=" * 70)
    print(f"PIPELINE COMPLETE in {pipeline_elapsed:.1f}s")
    print(f"TARGET_ENV: {TARGET_ENV} | CATALOG: {CATALOG}")
    print("=" * 70)

    # Show classification log for this run
    try:
        log_df = spark.sql(f"""
            SELECT table_name, staged_count, classified_count,
                   bhrut_deleted_count, unclassified_count, promoted_to_raw,
                   error_message
            FROM {CLASSIFICATION_LOG_TBL}
            WHERE run_timestamp >= '{RUN_TS.strftime("%Y-%m-%d %H:%M:%S")}'
            ORDER BY table_name
        """)
        if log_df.count() > 0:
            print("\nClassification Summary:")
            log_df.show(100, truncate=False)

            totals = spark.sql(f"""
                SELECT
                    SUM(staged_count) AS total_staged,
                    SUM(classified_count) AS total_classified,
                    SUM(bhrut_deleted_count) AS total_bhrut_deleted,
                    SUM(unclassified_count) AS total_unclassified,
                    SUM(promoted_to_raw) AS total_promoted
                FROM {CLASSIFICATION_LOG_TBL}
                WHERE run_timestamp >= '{RUN_TS.strftime("%Y-%m-%d %H:%M:%S")}'
            """).collect()[0]

            print(f"Totals: Staged={totals['total_staged']} | "
                  f"Classified={totals['total_classified']} | "
                  f"BHRUT Deleted={totals['total_bhrut_deleted']} | "
                  f"Unclassified={totals['total_unclassified']} | "
                  f"Promoted={totals['total_promoted']}")
        else:
            print("\nNo tables were processed in this run.")
    except Exception as e:
        print(f"\nCould not retrieve classification log: {e}")

    print(f"\nPipeline finished at {datetime.now()}")